In [5]:
# ============================================
# CELL 1 — Import Required Libraries
# ============================================

# pandas:
# المكتبة الأساسية التي سنستخدمها لقراءة وتنظيف وتحويل البيانات.
import pandas as pd

# numpy:
# نستخدمها في بعض عمليات الفحص والتعامل مع القيم الرقمية وNaN.
import numpy as np

# Path:
# تجعل التعامل مع مسارات الملفات أوضح وأسهل من كتابة Strings طويلة.
from pathlib import Path

# مكتبات Python Standard Library المستخدمة في التعامل مع الملفات.
import shutil
import os

# إعدادات عرض تساعدنا أثناء الشرح داخل Colab.
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


In [6]:
# ============================================
# CELL 4 — Define Project Paths
# ============================================

# المسار الرئيسي للمشروع بعد فك الضغط.
PROJECT_PATH = Path(r"D:\python vs\fintech_digital_payments_project_(1)")

# Raw Data:
# الملفات كما خرجت من السيستم قبل أي Cleaning.
RAW_DATA_PATH = PROJECT_PATH / "data" / "raw"

# Clean Data:
# سنصدر إليه النسخة النظيفة في نهاية مرحلة Python.
CLEAN_DATA_PATH = PROJECT_PATH / "data" / "clean"

# إنشاء مجلد clean إذا لم يكن موجودًا.
# CLEAN_DATA_PATH.mkdir(parents=True, exist_ok=True)

print("Project Path :", PROJECT_PATH)
print("Raw Data     :", RAW_DATA_PATH)
print("Clean Data   :", CLEAN_DATA_PATH)


Project Path : D:\python vs\fintech_digital_payments_project_(1)
Raw Data     : D:\python vs\fintech_digital_payments_project_(1)\data\raw
Clean Data   : D:\python vs\fintech_digital_payments_project_(1)\data\clean


In [7]:
# ============================================
# CELL 5 — Check Available Raw Files
# ============================================

# قبل قراءة الداتا، نتأكد أن الملفات التي نتوقعها موجودة بالفعل.
raw_files = sorted([file.name for file in RAW_DATA_PATH.glob("*.csv")])

print("Raw CSV files:")
for file_name in raw_files:
    print("-", file_name)

print(f"\nNumber of CSV files: {len(raw_files)}")

Raw CSV files:
- accounts.csv
- customers.csv
- locations.csv
- merchants.csv
- transaction_types.csv
- transactions.csv

Number of CSV files: 6


In [8]:
# ============================================
# CELL 6 — Load the 6 Raw Tables
# ============================================

customers = pd.read_csv(RAW_DATA_PATH / "customers.csv")
accounts = pd.read_csv(RAW_DATA_PATH / "accounts.csv")
merchants = pd.read_csv(RAW_DATA_PATH / "merchants.csv")
transaction_types = pd.read_csv(RAW_DATA_PATH / "transaction_types.csv")
locations = pd.read_csv(RAW_DATA_PATH / "locations.csv")

# low_memory=False يمنع pandas من تخمين أنواع البيانات على أجزاء صغيرة
# من جدول Transactions الكبير وإظهار تحذيرات غير ضرورية.
transactions = pd.read_csv(
    RAW_DATA_PATH / "transactions.csv",
    low_memory=False
)

print("All tables loaded successfully.")


All tables loaded successfully.


In [9]:
# ============================================
# CELL 7 — Create a Tables Dictionary
# ============================================

# بدل تكرار نفس الكود 6 مرات، نجمع الـ DataFrames داخل Dictionary.
# المفتاح = اسم الجدول
# القيمة = الـ DataFrame نفسه

tables = {
    "customers": customers,
    "accounts": accounts,
    "merchants": merchants,
    "transaction_types": transaction_types,
    "locations": locations,
    "transactions": transactions
}

print("Tables dictionary created.")


Tables dictionary created.


In [10]:
# ============================================
# CELL 8 — Dataset Size Summary
# ============================================

# نعمل Summary سريع يوضح حجم كل جدول.
table_summary = []

for table_name, df in tables.items():
    table_summary.append({
        "table_name": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

table_summary = pd.DataFrame(table_summary)

table_summary


,table_name,rows,columns
0,customers,45000,11
1,accounts,48000,8
2,merchants,5000,9
3,transaction_types,12,8
4,locations,180,5
5,transactions,400000,12


In [11]:
# ============================================
# CELL 9 — Total Number of Rows
# ============================================

# الهدف هنا إعطاء الطالب إحساس بحجم المشروع بالكامل.
total_rows = table_summary["rows"].sum()

print(f"Total rows across all tables: {total_rows:,}")

Total rows across all tables: 498,192


In [12]:
# CELL 10 — Preview Every Table
# ============================================

# نعرض أول 3 صفوف فقط من كل جدول.
# لا نستخدم display للـ 400K rows بالكامل بالطبع.

for table_name, df in tables.items():
    print("=" * 80)
    print("=" * 80)
    print(f"TABLE: {table_name.upper()}")
    print(f"Shape: {df.shape}")
    display(df.head(3))
# ============================================


TABLE: CUSTOMERS
Shape: (45000, 11)


,customer_id,first_name,last_name,gender,birth_year,customer_segment,kyc_status,signup_date,acquisition_channel,home_governorate,home_city
0,C000001,Farah,Salah,Female,2003,Youth,Verified,2022-12-22,Organic,Assiut,Assiut - Central
1,C000002,Sherif,Mahmoud,Male,1981,Mass,Verified,2023-12-27,Organic,Sharqia,10th of Ramadan
2,C000003,Ahmed,Helmy,Male,1998,Youth,Verified,2022-12-30,Agent Network,Monufia,Menouf


TABLE: ACCOUNTS
Shape: (48000, 8)


,account_id,customer_id,account_type,wallet_tier,account_status,opened_date,daily_limit_egp,current_balance_egp
0,A000001,C025947,Personal Wallet,Basic,Active,2022-06-24,"10,000.00",741.42
1,A000002,C016973,Business Wallet,Business,Active,2022-03-03,"80,000.00","19,581.52"
2,A000003,C003186,Personal Wallet,Basic,Active,2024-09-22,"10,000.00","1,212.34"


TABLE: MERCHANTS
Shape: (5000, 9)


...

,transaction_type_id,transaction_type_en,transaction_type_ar,category,fee_model,base_fee_egp,variable_fee_rate,merchant_required
0,TT01,Merchant Payment,دفع لدى تاجر,Payment,Percent,0.00,0.01,1
1,TT02,QR Payment,دفع QR,Payment,Percent,0.00,0.01,1
2,TT03,Online Checkout,دفع أونلاين,Payment,Percent,0.00,0.02,1


TABLE: LOCATIONS
Shape: (180, 5)


,location_id,governorate,city_zone,region,area_type
0,L0001,Cairo,Nasr City,Greater Cairo,Urban
1,L0002,Cairo,Heliopolis,Greater Cairo,Urban
2,L0003,Cairo,Maadi,Greater Cairo,Urban


TABLE: TRANSACTIONS
Shape: (400000, 12)


,transaction_id,transaction_datetime,account_id,merchant_id,transaction_type_id,location_id,transaction_amount_egp,provider_fee_egp,transaction_status,payment_channel,device_type,processing_time_seconds
0,TX000000001,2025-12-15 10:18:59,A019198,NaN,TT07,L0054,433.42,1.50,Success,Mobile App,Android,2.09
1,TX000000002,2025-06-09 15:47:56,A034587,NaN,TT05,L0172,2162.68,0.00,Success,Agent,Agent Terminal,2.03
2,TX000000003,2026-03-20 19:20:20,A018300,NaN,TT12,L0097,788.74,4.73,Success,Mobile App,iOS,1.29


In [13]:
# ============================================
# CELL 11 — Inspect Data Types
# ============================================

# Data Types مهمة جدًا قبل Cleaning.
# مثال:
# Date قد تظهر Object.
# Numeric column قد تظهر Object بسبب commas أو spaces.

for table_name, df in tables.items():
    print("=" * 80)
    print(f"TABLE: {table_name.upper()}")
    print(df.dtypes)
    print()


TABLE: CUSTOMERS
customer_id              str
first_name               str
last_name                str
gender                   str
birth_year             int64
customer_segment         str
kyc_status               str
signup_date              str
acquisition_channel      str
home_governorate         str
home_city                str
dtype: object

TABLE: ACCOUNTS
account_id                 str
customer_id                str
account_type               str
wallet_tier                str
account_status             str
opened_date                str
daily_limit_egp        float64
current_balance_egp    float64
dtype: object

TABLE: MERCHANTS
merchant_id               str
merchant_name             str
merchant_category         str
merchant_size             str
onboarding_date           str
merchant_status           str
settlement_cycle          str
registered_governorate    str
registered_city           str
dtype: object

TABLE: TRANSACTION_TYPES
transaction_type_id        str
transaction_

In [14]:
# ============================================
# CELL 12 — Missing Values Summary
# ============================================

# نحسب عدد ونسبة الـ Missing Values في كل Column.
# لا نحذف أي Missing Values الآن.
# الأول نفهم: هل الـ Null مشكلة Data Quality أم له Business Meaning؟

for table_name, df in tables.items():
    missing_count = df.isna().sum()
    missing_pct = (missing_count / len(df) * 100).round(2)

    missing_summary = pd.DataFrame({
        "missing_count": missing_count,
        "missing_pct": missing_pct
    })

    # نعرض فقط الأعمدة التي تحتوي على Missing Values.
    missing_summary = missing_summary[missing_summary["missing_count"] > 0]

    print("=" * 80)
    print(f"TABLE: {table_name.upper()}")

    if missing_summary.empty:
        print("No missing values detected.")
    else:
        display(missing_summary)


TABLE: CUSTOMERS


,missing_count,missing_pct
acquisition_channel,315,0.70
home_city,225,0.50


TABLE: ACCOUNTS
No missing values detected.
TABLE: MERCHANTS


,missing_count,missing_pct
settlement_cycle,20,0.40


TABLE: TRANSACTION_TYPES
No missing values detected.
TABLE: LOCATIONS
No missing values detected.
TABLE: TRANSACTIONS


,missing_count,missing_pct
merchant_id,199528,49.88
processing_time_seconds,1600,0.40


In [15]:
# ============================================
# CELL 13 — Primary Key Uniqueness Check
# ============================================

primary_keys = {
    "customers": "customer_id",
    "accounts": "account_id",
    "merchants": "merchant_id",
    "transaction_types": "transaction_type_id",
    "locations": "location_id",
    "transactions": "transaction_id"
}

pk_results = []

for table_name, key_column in primary_keys.items():
    df = tables[table_name]

    pk_results.append({
        "table_name": table_name,
        "primary_key": key_column,
        "rows": len(df),
        "unique_values": df[key_column].nunique(),
        "duplicate_rows": df[key_column].duplicated().sum(),
        "missing_keys": df[key_column].isna().sum()
    })

pk_results = pd.DataFrame(pk_results)

pk_results


,table_name,primary_key,rows,unique_values,duplicate_rows,missing_keys
0,customers,customer_id,45000,45000,0,0
1,accounts,account_id,48000,48000,0,0
2,merchants,merchant_id,5000,5000,0,0
3,transaction_types,transaction_type_id,12,12,0,0
4,locations,location_id,180,180,0,0
5,transactions,transaction_id,400000,400000,0,0


In [16]:
# ============================================
# CELL 14 — Foreign Key Integrity Check
# ============================================

# Accounts -> Customers
accounts_without_customer = (
    ~accounts["customer_id"].isin(customers["customer_id"])
).sum()

# Transactions -> Accounts
transactions_without_account = (
    ~transactions["account_id"].isin(accounts["account_id"])
).sum()

# Transactions -> Transaction Types
transactions_without_type = (
    ~transactions["transaction_type_id"].isin(
        transaction_types["transaction_type_id"]
    )
).sum()

# Transactions -> Locations
transactions_without_location = (
    ~transactions["location_id"].isin(locations["location_id"])
).sum()

# merchant_id قد يكون Null بشكل صحيح لبعض أنواع المعاملات.
# لذلك نفحص فقط القيم غير الفارغة.
non_null_merchant = transactions["merchant_id"].dropna()

transactions_without_merchant = (
    ~non_null_merchant.isin(merchants["merchant_id"])
).sum()

fk_check = pd.DataFrame({
    "relationship": [
        "accounts.customer_id -> customers.customer_id",
        "transactions.account_id -> accounts.account_id",
        "transactions.transaction_type_id -> transaction_types.transaction_type_id",
        "transactions.location_id -> locations.location_id",
        "transactions.merchant_id -> merchants.merchant_id"
    ],
    "orphan_records": [
        accounts_without_customer,
        transactions_without_account,
        transactions_without_type,
        transactions_without_location,
        transactions_without_merchant
    ]
})

fk_check


,relationship,orphan_records
0,accounts.customer_id -> customers.customer_id,0
1,transactions.account_id -> accounts.account_id,0
2,transactions.transaction_type_id -> transactio...,0
3,transactions.location_id -> locations.location_id,0
4,transactions.merchant_id -> merchants.merchant_id,0


In [17]:
# ============================================
# CELL 15 — Inspect Important Categorical Values
# ============================================

categorical_checks = {
    "customers.customer_segment": customers["customer_segment"],
    "customers.kyc_status": customers["kyc_status"],
    "accounts.wallet_tier": accounts["wallet_tier"],
    "accounts.account_status": accounts["account_status"],
    "merchants.merchant_category": merchants["merchant_category"],
    "transactions.transaction_status": transactions["transaction_status"],
    "transactions.payment_channel": transactions["payment_channel"]
}

for column_name, series in categorical_checks.items():
    print("=" * 80)
    print(column_name)
    print(series.value_counts(dropna=False).head(20))
    print()


customers.customer_segment
customer_segment
Mass               25187
Youth              12416
Affluent            5432
Microbusiness       1785
 mass                 60
MASS                  49
YOUTH                 24
 youth                24
 affluent             11
AFFLUENT               8
 microbusiness         3
MICROBUSINESS          1
Name: count, dtype: int64

customers.kyc_status
kyc_status
Verified          42956
Pending Review     1581
Restricted          463
Name: count, dtype: int64

accounts.wallet_tier
wallet_tier
Basic         25926
Silver        13952
Gold           6395
Business       1535
 basic          100
 silver          61
 gold            25
 business         6
Name: count, dtype: int64

accounts.account_status
account_status
Active       44844
Suspended     2189
Closed         967
Name: count, dtype: int64

merchants.merchant_category
merchant_category
Grocery             942
Restaurants         715
Transportation      502
Pharmacy            500
E-commerce   

In [18]:
# ============================================
# CELL 16 — Inspect Transaction Data Types More Closely
# ============================================

transaction_columns_to_check = [
    "transaction_datetime",
    "transaction_amount_egp",
    "provider_fee_egp",
    "processing_time_seconds"
]

transactions[transaction_columns_to_check].dtypes


transaction_datetime           str
transaction_amount_egp         str
provider_fee_egp           float64
processing_time_seconds    float64
dtype: object

In [19]:
# ============================================
# CELL 17 — Sample Suspicious Transaction Amount Values
# ============================================

# نحول العمود إلى String بشكل مؤقت للفحص فقط.
# نبحث عن قيم تحتوي على comma أو leading/trailing spaces.

amount_as_text = transactions["transaction_amount_egp"].astype(str)

suspicious_amounts = transactions[
    amount_as_text.str.contains(",", regex=False)
    | amount_as_text.str.startswith(" ")
    | amount_as_text.str.endswith(" ")
][["transaction_id", "transaction_amount_egp"]]

print(f"Suspicious amount formatting rows: {len(suspicious_amounts):,}")

suspicious_amounts.head(10)

Suspicious amount formatting rows: 998


,transaction_id,transaction_amount_egp
561,TX000000562,343.74
685,TX000000686,177.84
1164,TX000001165,"3,346.79"
1210,TX000001211,"2,248.43"
1351,TX000001352,311.17
1432,TX000001433,"2,241.59"
2248,TX000002249,"1,709.27"
2305,TX000002306,328.50
2406,TX000002407,"3,069.43"
2482,TX000002483,"1,234.88"


In [20]:
# ============================================
# CELL 18 — Sample Date Formats
# ============================================

# نعرض Sample من DateTime كما جاءت من السيستم.
# الهدف ملاحظة أن العمود يحتوي على أكثر من Format.

transactions["transaction_datetime"].sample(
    15,
    random_state=42
).tolist()

['21/09/2025 13:34:36',
 '2025-10-19 23:27:36',
 '2026-05-04 14:48:04',
 '2025-02-24 17:58:24',
 '2025-03-17 11:45:08',
 '2025-05-18 13:18:42',
 '2026-05-29 18:45:32',
 '2025-11-24 10:43:44',
 '2026-03-26 05:58:56',
 '2025-07-31 04:50:28',
 '2025-07-21 09:46:22',
 '2025-11-30 09:47:44',
 '2026-06-27 14:26:41',
 '2025-05-05 22:04:51',
 '2025-05-19 14:16:07']

In [21]:
# ============================================
# CELL 19 — Create a Clean Copy of Customers
# ============================================

# نحتفظ بالـ Raw DataFrame بدون تعديل.
customers_clean = customers.copy()

print(customers_clean.shape)


(45000, 11)


In [22]:
# ============================================
# CELL 20 — Normalize Customer Text Fields
# ============================================

# customer_segment يحتوي على اختلافات مثل:
# " mass "
# "MASS"
# "Mass"
#
# strip() تزيل الـ spaces من البداية والنهاية.
# title() توحد طريقة كتابة الكلمات.

customers_clean["customer_segment"] = (
    customers_clean["customer_segment"]
    .astype("string")
    .str.strip()
    .str.title()
)

# تنظيف بقية الحقول النصية الأساسية.
text_columns = [
    "first_name",
    "last_name",
    "gender",
    "kyc_status",
    "acquisition_channel",
    "home_governorate",
    "home_city"
]

for col in text_columns:
    customers_clean[col] = customers_clean[col].astype("string").str.strip()

In [23]:
# ============================================
# CELL 21 — Convert Customer Date
# ============================================

customers_clean["signup_date"] = pd.to_datetime(
    customers_clean["signup_date"],
    errors="coerce"
)

print(customers_clean["signup_date"].dtype)


datetime64[us]


In [24]:
# ============================================
# CELL 22 — Handle Missing Customer Descriptions
# ============================================

# لا نحذف العملاء بسبب Missing descriptive attribute.
# نستخدم "Unknown" لكي لا نفقد Transaction history مرتبطة بالعميل.

customers_clean["acquisition_channel"] = (
    customers_clean["acquisition_channel"]
    .fillna("Unknown")
)

customers_clean["home_city"] = (
    customers_clean["home_city"]
    .fillna("Unknown")
)


In [25]:
# ============================================
# CELL 23 — Clean Accounts
# ============================================

accounts_clean = accounts.copy()

# Normalize text.
accounts_clean["wallet_tier"] = (
    accounts_clean["wallet_tier"]
    .astype("string")
    .str.strip()
    .str.title()
)

for col in ["account_type", "account_status"]:
    accounts_clean[col] = (
        accounts_clean[col]
        .astype("string")
        .str.strip()
    )

# Convert date.
accounts_clean["opened_date"] = pd.to_datetime(
    accounts_clean["opened_date"],
    errors="coerce"
)

# Convert numeric columns explicitly.
accounts_clean["daily_limit_egp"] = pd.to_numeric(
    accounts_clean["daily_limit_egp"],
    errors="coerce"
)

accounts_clean["current_balance_egp"] = pd.to_numeric(
    accounts_clean["current_balance_egp"],
    errors="coerce"
)

accounts_clean.head()


,account_id,customer_id,account_type,wallet_tier,account_status,opened_date,daily_limit_egp,current_balance_egp
0,A000001,C025947,Personal Wallet,Basic,Active,2022-06-24,"10,000.00",741.42
1,A000002,C016973,Business Wallet,Business,Active,2022-03-03,"80,000.00","19,581.52"
2,A000003,C003186,Personal Wallet,Basic,Active,2024-09-22,"10,000.00","1,212.34"
3,A000004,C011240,Personal Wallet,Silver,Active,2025-02-03,"20,000.00","5,008.01"
4,A000005,C006833,Personal Wallet,Silver,Active,2023-08-12,"20,000.00","1,361.02"


In [26]:
# ============================================
# CELL 24 — Clean Merchants
# ============================================

merchants_clean = merchants.copy()

merchants_clean["merchant_category"] = (
    merchants_clean["merchant_category"]
    .astype("string")
    .str.strip()
    .str.title()
)

for col in [
    "merchant_name",
    "merchant_size",
    "merchant_status",
    "settlement_cycle",
    "registered_governorate",
    "registered_city"
]:
    merchants_clean[col] = (
        merchants_clean[col]
        .astype("string")
        .str.strip()
    )

merchants_clean["onboarding_date"] = pd.to_datetime(
    merchants_clean["onboarding_date"],
    errors="coerce"
)

# Missing Settlement Cycle لا يستدعي حذف Merchant.
merchants_clean["settlement_cycle"] = (
    merchants_clean["settlement_cycle"]
    .fillna("Unknown")
)

merchants_clean.head()


,merchant_id,merchant_name,merchant_category,merchant_size,onboarding_date,merchant_status,settlement_cycle,registered_governorate,registered_city
0,M00001,Family Care 0001,Pharmacy,Micro,2022-03-09,Active,T+1,Cairo,Ain Shams - Central
1,M00002,Cairo Tech 0002,Electronics,Micro,2021-09-23,Active,T+0,Alexandria,Sidi Gaber - Central
2,M00003,Delta Academy 0003,Education,Small,2025-11-13,Active,T+1,Cairo,Maadi - Central
3,M00004,Nile Market 0004,Grocery,Medium,2023-02-07,Active,T+2,Kafr El Sheikh,Kafr El Sheikh - Central
4,M00005,Cairo Pharmacy 0005,Pharmacy,Micro,2021-11-19,Active,T+1,Alexandria,Stanley - Central


In [27]:
# ============================================
# CELL 25 — Clean Transaction Types & Locations
# ============================================

transaction_types_clean = transaction_types.copy()
locations_clean = locations.copy()

# Reference tables بالفعل نظيفة نسبيًا،
# لكن نطبق strip على النصوص كـ Defensive Cleaning.

for col in transaction_types_clean.select_dtypes(include="object").columns:
    transaction_types_clean[col] = (
        transaction_types_clean[col]
        .astype("string")
        .str.strip()
    )

for col in locations_clean.select_dtypes(include="object").columns:
    locations_clean[col] = (
        locations_clean[col]
        .astype("string")
        .str.strip()
    )

print("Reference tables cleaned.")


Reference tables cleaned.


C:\Users\Admin\AppData\Local\Temp\ipykernel_6328\924158287.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in transaction_types_clean.select_dtypes(include="object").columns:
C:\Users\Admin\AppData\Local\Temp\ipykernel_6328\924158287.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.o

In [28]:
# ============================================
# CELL 26 — Create Transactions Clean Copy
# ============================================

transactions_clean = transactions.copy()

print(transactions_clean.shape)

(400000, 12)


In [31]:
# ============================================
# CELL 27 — Clean Transaction Amount
# ============================================

# بعض القيم خرجت من السيستم كنص بسبب وجود:
# commas
# spaces
#
# مثال:
# "1,250.50"
# " 950.00 "

transactions_clean["transaction_amount_egp"] = (
    transactions_clean["transaction_amount_egp"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.strip()
)

# بعد تنظيف النص نحوله إلى رقم.
transactions_clean["transaction_amount_egp"] = pd.to_numeric(
    transactions_clean["transaction_amount_egp"],
    errors="coerce"
)


In [32]:
# ============================================
# CELL 28 — Clean Transaction DateTime
# ============================================

# العمود يحتوي على Mixed Date Formats.
#
# format="mixed":
# يسمح لـ pandas بالتعامل مع أكثر من شكل للتاريخ.
#
# dayfirst=True:
# مهم للـ DD/MM/YYYY rows الموجودة في الـ Raw Export.

transactions_clean["transaction_datetime"] = pd.to_datetime(
    transactions_clean["transaction_datetime"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

print(transactions_clean["transaction_datetime"].dtype)


datetime64[us]


In [33]:
# ============================================
# CELL 29 — Normalize Transaction Status
# ============================================

transactions_clean["transaction_status"] = (
    transactions_clean["transaction_status"]
    .astype("string")
    .str.strip()
    .str.title()
)

transactions_clean["transaction_status"].value_counts(dropna=False)


transaction_status
Success     375896
Failed       18267
Reversed      3368
Pending       2469
Name: count, dtype: Int64

In [34]:
# ============================================
# CELL 30 — Normalize Payment Channel
# ============================================

# أولًا نوحد النص لتسهيل عملية Mapping.
channel_normalized = (
    transactions_clean["payment_channel"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace("_", " ", regex=False)
)

# Mapping يحول كل Variant إلى Business Label موحدة.
channel_mapping = {
    "mobile app": "Mobile App",
    "web": "Web",
    "pos": "POS",
    "qr": "QR",
    "api": "API",
    "agent": "Agent",
    "atm": "ATM",
    "ussd": "USSD"
}

transactions_clean["payment_channel"] = (
    channel_normalized.map(channel_mapping)
)

transactions_clean["payment_channel"].value_counts(dropna=False)


payment_channel
Mobile App    169100
POS            69193
QR             54580
Web            43627
Agent          39513
API            14688
ATM             5894
USSD            3405
Name: count, dtype: int64

In [35]:
# ============================================
# CELL 31 — Convert Remaining Numeric Columns
# ============================================

transactions_clean["provider_fee_egp"] = pd.to_numeric(
    transactions_clean["provider_fee_egp"],
    errors="coerce"
)

transactions_clean["processing_time_seconds"] = pd.to_numeric(
    transactions_clean["processing_time_seconds"],
    errors="coerce"
)


In [36]:

# ============================================
# CELL 32 — Create Useful Time Attributes
# ============================================

# Python هنا لا تحسب KPI.
# نحن فقط نستخرج Attributes مفيدة للتحليل التشغيلي.

transactions_clean["transaction_date"] = (
    transactions_clean["transaction_datetime"].dt.date
)

transactions_clean["transaction_hour"] = (
    transactions_clean["transaction_datetime"].dt.hour
)

transactions_clean[[
    "transaction_datetime",
    "transaction_date",
    "transaction_hour"
]].head()


,transaction_datetime,transaction_date,transaction_hour
0,2025-12-15 10:18:59,2025-12-15,10
1,2025-09-06 15:47:56,2025-09-06,15
2,2026-03-20 19:20:20,2026-03-20,19
3,2026-02-27 20:41:34,2026-02-27,20
4,2026-06-16 20:24:15,2026-06-16,20


In [37]:
# ============================================
# CELL 33 — Rebuild Clean Tables Dictionary
# ============================================

clean_tables = {
    "customers": customers_clean,
    "accounts": accounts_clean,
    "merchants": merchants_clean,
    "transaction_types": transaction_types_clean,
    "locations": locations_clean,
    "transactions": transactions_clean
}


In [38]:
# ============================================
# CELL 34 — Missing Values After Cleaning
# ============================================

for table_name, df in clean_tables.items():
    missing_count = df.isna().sum()
    missing_pct = (missing_count / len(df) * 100).round(2)

    result = pd.DataFrame({
        "missing_count": missing_count,
        "missing_pct": missing_pct
    })

    result = result[result["missing_count"] > 0]

    print("=" * 80)
    print(table_name.upper())

    if result.empty:
        print("No missing values.")
    else:
        display(result)


CUSTOMERS
No missing values.
ACCOUNTS
No missing values.
MERCHANTS
No missing values.
TRANSACTION_TYPES
No missing values.
LOCATIONS
No missing values.
TRANSACTIONS


,missing_count,missing_pct
merchant_id,199528,49.88
processing_time_seconds,1600,0.40


In [39]:
# ============================================
# CELL 35 — Numeric Business Rules
# ============================================

invalid_amount = (
    transactions_clean["transaction_amount_egp"] <= 0
).sum()

negative_fee = (
    transactions_clean["provider_fee_egp"] < 0
).sum()

print(f"Transactions with amount <= 0 : {invalid_amount:,}")
print(f"Transactions with negative fee: {negative_fee:,}")


Transactions with amount <= 0 : 0
Transactions with negative fee: 0


In [40]:
# ============================================
# CELL 36 — Validate Parsed Date Range
# ============================================

print("Min transaction datetime:",
      transactions_clean["transaction_datetime"].min())

print("Max transaction datetime:",
      transactions_clean["transaction_datetime"].max())

print(
    "Unparsed transaction dates:",
    transactions_clean["transaction_datetime"].isna().sum()
)


Min transaction datetime: 2025-01-01 00:01:09
Max transaction datetime: 2026-12-06 23:59:54
Unparsed transaction dates: 0


In [41]:
# ============================================
# CELL 37 — Validate Business-Valid Merchant Nulls
# ============================================

# merchant_id ليس مطلوبًا في كل أنواع المعاملات.
# نضم Transactions مع Transaction Types للحصول على merchant_required.

merchant_validation = transactions_clean[
    ["transaction_id", "transaction_type_id", "merchant_id"]
].merge(
    transaction_types_clean[
        ["transaction_type_id", "merchant_required"]
    ],
    on="transaction_type_id",
    how="left"
)

# المشكلة الحقيقية فقط:
# merchant_required = 1 ولكن merchant_id = Null
invalid_missing_merchant = merchant_validation[
    (merchant_validation["merchant_required"] == 1)
    & (merchant_validation["merchant_id"].isna())
]

print(
    "Invalid missing merchant rows:",
    len(invalid_missing_merchant)
)


Invalid missing merchant rows: 0


In [42]:
# ============================================
# CELL 38 — Final Primary Key Check
# ============================================

for table_name, key_column in primary_keys.items():
    df = clean_tables[table_name]

    duplicates = df[key_column].duplicated().sum()
    missing = df[key_column].isna().sum()

    print(
        f"{table_name:<20} | "
        f"Duplicates: {duplicates:<5} | "
        f"Missing PK: {missing}"
    )


customers            | Duplicates: 0     | Missing PK: 0
accounts             | Duplicates: 0     | Missing PK: 0
merchants            | Duplicates: 0     | Missing PK: 0
transaction_types    | Duplicates: 0     | Missing PK: 0
locations            | Duplicates: 0     | Missing PK: 0
transactions         | Duplicates: 0     | Missing PK: 0


In [44]:
#  ============================================
# CELL 39 — Final Foreign Key Check
# ============================================

print(
    "Accounts without valid Customer:",
    (~accounts_clean["customer_id"].isin(
        customers_clean["customer_id"]
    )).sum()
)

print(
    "Transactions without valid Account:",
    (~transactions_clean["account_id"].isin(
        accounts_clean["account_id"]
    )).sum()
)

print(
    "Transactions without valid Type:",
    (~transactions_clean["transaction_type_id"].isin(
        transaction_types_clean["transaction_type_id"]
    )).sum()
)

print(
    "Transactions without valid Location:",
    (~transactions_clean["location_id"].isin(
        locations_clean["location_id"]
    )).sum()
)

merchant_values = transactions_clean["merchant_id"].dropna()

print(
    "Transactions with invalid Merchant:",
    (~merchant_values.isin(
        merchants_clean["merchant_id"]
    )).sum()
)


Accounts without valid Customer: 0
Transactions without valid Account: 0
Transactions without valid Type: 0
Transactions without valid Location: 0
Transactions with invalid Merchant: 0


In [45]:
# ============================================
# CELL 40 — Export the 6 Clean Tables
# ============================================

customers_clean.to_csv(
    CLEAN_DATA_PATH / "customers_clean.csv",
    index=False
)

accounts_clean.to_csv(
    CLEAN_DATA_PATH / "accounts_clean.csv",
    index=False
)

merchants_clean.to_csv(
    CLEAN_DATA_PATH / "merchants_clean.csv",
    index=False
)

transaction_types_clean.to_csv(
    CLEAN_DATA_PATH / "transaction_types_clean.csv",
    index=False
)

locations_clean.to_csv(
    CLEAN_DATA_PATH / "locations_clean.csv",
    index=False
)

transactions_clean.to_csv(
    CLEAN_DATA_PATH / "transactions_clean.csv",
    index=False
)

print("All clean tables exported successfully.")


All clean tables exported successfully.


In [46]:
# ============================================
# CELL 41 — Verify Exported Files
# ============================================

clean_files = sorted(
    [file.name for file in CLEAN_DATA_PATH.glob("*.csv")]
)

for file_name in clean_files:
    print("-", file_name)

print(f"\nNumber of clean files: {len(clean_files)}")


- accounts_clean.csv
- customers_clean.csv
- locations_clean.csv
- merchants_clean.csv
- transaction_types_clean.csv
- transactions_clean.csv

Number of clean files: 6


In [47]:
# ============================================
# CELL 42 — ZIP Clean Data for Power BI
# ============================================

clean_zip_base = "/content/fintech_clean_data"

shutil.make_archive(
    clean_zip_base,
    "zip",
    CLEAN_DATA_PATH
)

print("ZIP created:", clean_zip_base + ".zip")


ZIP created: /content/fintech_clean_data.zip
